<a href="https://colab.research.google.com/github/parthag1201/RAG-ify/blob/main/rag_from_scratch_P1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1 : Implementing RAG Pipeline using langchain , langsmith for monitoring dashboard and Google Gemini API

In [ ]:
# (1) Install required packages (if missing)
! pip install google-generativeai langchain_google_genai chromadb langchain
! pip install langchain_community tiktoken langchain-openai langchainhub

In [2]:
# (2) Import Gemini components
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

In [3]:
from google.colab import userdata # For API Secret

In [4]:
# LangSmith Configuration
import os
from langsmith import traceable   ## To use @traceable on llm calls
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = userdata.get('LANGCHAIN_API_KEY')
# LANGCHAIN_API_KEY = userdata.get('LANGCHAIN_API_KEY')
os.environ['LANGSMITH_PROJECT']='Rag-from-scratch_P1'

In [ ]:
from langsmith import utils
utils.tracing_is_enabled()

In [ ]:
# LangChain Libraries
import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [7]:
from google.colab import auth
auth.authenticate_user()

In [8]:
# (3) Configure API keys
import os
os.environ['GOOGLE_API_KEY']=userdata.get('Gemini_API')
# GOOGLE_API_KEY = userdata.get('Gemini_API')  # Replace with actual key

# (4) Initialize Gemini components
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

In [ ]:
print(docs)

In [ ]:
# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

print(splits)

In [11]:
# Rest of the code remains same until vectorstore initialization
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,  # Using Gemini embeddings
    # persist_directory="./chrome_db"
)

retriever = vectorstore.as_retriever()

In [12]:
from langchain.prompts import ChatPromptTemplate, PromptTemplate  # Import from correct submodule
from langchain import LLMChain

In [13]:
# (5) Update prompt template for Gemini compatibility
prompt_template = """Answer the question based only on the context:
Context: {context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(prompt_template)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# (6) Create RAG chain with Gemini
# traceable
# def rag(context,question,prompt,llm):
#   rag_chain = (
#     {"context": retriever | format_docs, "question": RunnablePassthrough()}
#     | prompt
#     | llm
#     | StrOutputParser()
#   )

# # Query execution remains same
#   rag_chain.invoke("What is a Task?")

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
  )

rag_chain.invoke("What is a Task?")

# Part 2 : Indexing

In [15]:
# Doc
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

In [19]:
# ENCODING STRING : " To convert number into a vector array"

## Get the number of tokens of a doc
import tiktoken

def num_tokens_from_string(string, encoding_name) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    # print(encoding.encode(string))
    return num_tokens

num_tokens_from_string(question, "cl100k_base")

[3923, 13124, 315, 26159, 656, 358, 1093, 30]


8

In [ ]:
# EMBEDDING

## Why ? Because embedding is basically converting text to numerical vector representations to capture a closer semantic meaning of the text.
## LLMs can't understand long sentences easily that's why we will split them into smaller chunks and then put them in a vector space.
## We can retrive these vectors (mapped according to indexes) using semantic of the prompts later.

embd = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
query_result = embd.embed_query(question)
document_result = embd.embed_query(document)
# print(query_result)
len(query_result)

In [22]:
# COSINE SIMILARITY : "Used after embedding to find most relevant matches from the created vector spaces"

## Vectors pointing in same direction will be having same context - even if words aren't the same
## Cosine_Output = A (Dot) B / |A|*|B|

import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)

Cosine Similarity: 0.8535652119095083


In [24]:
#### INDEXING ####

# Load blog using LangChain Document Loader
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

In [ ]:
# SPLIT : "We try to split on them in order until the chunks are small enough. The default list is ["\n\n", "\n", " ", ""]"
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(blog_docs)
# print(splits)

In [27]:
# Indexing : Into the vector store and retriever
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=GoogleGenerativeAIEmbeddings(model="models/embedding-001"))

retriever = vectorstore.as_retriever()

# Part 3 : Retrieval

In [28]:
# Index
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=GoogleGenerativeAIEmbeddings(model="models/embedding-001"))


retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

In [42]:
docs = retriever.invoke("What is Task Decomposition?")

In [ ]:
print(docs)

In [31]:
len(docs)

1

# Part 4 : Generation

In [ ]:
# PROMPT

# Required packages :
# import google.generativeai as genai
# from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

In [39]:
# LLM
llm = llm

In [40]:
# Chain
chain = prompt | llm

In [ ]:
# Run
chain.invoke({"context":docs,"question":"What is Task Decomposition?"})

In [ ]:
# Sample rag-prompt with placeholders for context and question
from langchain import hub
prompt_hub_rag = hub.pull("rlm/rag-prompt")
prompt_hub_rag

In [ ]:
# CHAIN RUNNABLES

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")